# Step 13 (end-to-end) — Pretrained EDM on CIFAR-10: is there a linear collective variable that carries the transition?

Responds to review points 8 and 9. Self-contained; every cell can be re-run after a kernel restart provided the cells above it have run.

**Setting.** NVIDIA's public EDM checkpoint `edm-cifar10-32x32-uncond-vp.pkl` (DDPM++, 56M parameters). For an unconditional model the exact reference along any direction $u$ is the **full CIFAR-10 training set** projected on $u$.

**A.** Eleven LDA directions along which the whole distribution could be two lumps (vehicles vs animals; each class vs rest): bisected $\tau_{\rm bif}$, cusp fit, and the Fisher separability that explains whether a transition exists. Optionally the balanced-pair census on pair subsets (data only, comparable to MNIST; off by default since it has been run).

**B.** The model's own soft modes (top Jacobian eigenvector of the pretrained denoiser at several $\sigma$): is the data bimodal along them, and if so, does the network reproduce the transition?

Internet on (clones `NVlabs/edm`, downloads ~200 MB). GPU on. ~20 min.

In [ ]:
RUN_PAIR_CENSUS = False
import os, sys, subprocess, pickle, numpy as np, torch, math, time
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
from scipy.signal import fftconvolve
from scipy.stats import theilslopes
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 130
if not os.path.exists('/kaggle/working/edm'): subprocess.run(['git', 'clone', '-q', 'https://github.com/NVlabs/edm.git', '/kaggle/working/edm'], check=True)
sys.path.insert(0, '/kaggle/working/edm'); import dnnlib
d = datasets.CIFAR10('/kaggle/working/cifar', train=True, download=True)
X = (np.asarray(d.data).astype(np.float32)/127.5 - 1.0).transpose(0, 3, 1, 2).reshape(-1, 3072); Y = np.asarray(d.targets); print(X.shape)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = [0, 1, 8, 9]; ANI = [2, 3, 4, 5, 6, 7]

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True); k = np.exp(-GRID**2/(2*tau)); k /= k.sum()
    return np.maximum(fftconvolve(h, k, mode='same'), 1e-300)
def exact_drift(m0, tau): p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(p):
    d_ = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d_[:-1]) != np.sign(d_[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d_[i]*DG/(d_[i+1]-d_[i])) if d_[i+1] != d_[i] else float(GRID[i]), 'max' if d_[i] > 0 else 'min') for i in idx])
def n_max(pts): return sum(k == 'max' for _, k in pts)
def birth_bisect(get_pts, taus=np.geomspace(0.01, 3.0, 80), iters=10):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b, clamped=(-c3 <= b_min+1e-12))
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(Xa.shape[1]), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def zero_crossings(cen, mean, cnt, min_count=60):
    okk = cnt >= min_count; c, f = cen[okk], mean[okk]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0: out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return _merge(out, merge=0.05)

## A. LDA directions of the full distribution, with separability diagnosis

In [ ]:
def analyse_direction(Xa, Xb, Xfull, label):
    u, mid, half = lda_direction(Xa, Xb); pa, pb = (Xa-mid) @ u, (Xb-mid) @ u
    sep = abs(pa.mean()-pb.mean())/np.sqrt(0.5*(pa.var()+pb.var())); m0 = (Xfull - mid) @ u / half
    modes_raw = n_max(stationary_points(smooth_hist(m0, 0.01)))
    bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
    line = f"{label:>22}: Fisher sep {sep:.2f}, modes of raw projection {modes_raw}, "
    if bj is None: print(line + "never bimodal"); return None
    f = cusp_fit(*exact_drift(m0, bj[0])); print(line + f"tau_bif={bj[0]:.3f} jump={bj[1]:.2f} h={f['h']:+.4f} b={f['b']:.3f}{'  [b clamped]' if f['clamped'] else ''}")
    return dict(label=label, tau=bj[0], jump=bj[1], u=u, mid=mid, half=half, m0=m0, sep=sep, **f)

print("== Full CIFAR-10 distribution along 11 LDA directions (what the unconditional model sees) ==")
DIRS = [analyse_direction(X[np.isin(Y, VEH)], X[np.isin(Y, ANI)], X, 'vehicles vs animals')]
for c in range(10): DIRS.append(analyse_direction(X[Y == c], X[Y != c], X, f'{CLASSES[c]} vs rest'))
DIRS = [r for r in DIRS if r]

if RUN_PAIR_CENSUS:
    print("\n== Balanced-pair census on pair subsets (data only) ==")
    PAIRS = []
    for a_, b_ in combinations(range(10), 2):
        Xa, Xb = X[Y == a_], X[Y == b_]; u, mid, half = lda_direction(Xa, Xb); m0 = np.concatenate([(Xa-mid) @ u, (Xb-mid) @ u]) / half
        bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
        if bj: PAIRS.append(dict(pair=(a_, b_), tau=bj[0], jump=bj[1], **cusp_fit(*exact_drift(m0, bj[0]))))
    H = np.array([abs(r['h']) for r in PAIRS]); nd = np.array([not r['clamped'] and r['tau'] >= 0.45 for r in PAIRS])
    print(f"balanced pairs: {len(PAIRS)}/45 bimodal; |h| median {np.median(H):.4f}; non-degenerate (b not clamped, tau_bif>=0.45): {nd.sum()}")

## B. The pretrained model's own soft modes

In [ ]:
URL = 'https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl'
with dnnlib.util.open_url(URL) as f: net = pickle.load(f)['ema'].to(dev).eval()
print('loaded EDM CIFAR-10 unconditional; sigma range', net.sigma_min, net.sigma_max)

@torch.no_grad()
def soft_mode(sigma, n=128, iters=12, eps=5e-2, bs=64):
    x0 = torch.from_numpy(X[rng.choice(len(X), n)]).to(dev).view(-1, 3, 32, 32); xt = x0 + sigma*torch.randn_like(x0)
    v = torch.randn(3072, device=dev); v /= v.norm(); lam = 0.0
    for _ in range(iters):
        Jv = torch.zeros(3072, device=dev)
        for xb in xt.split(bs):
            s = torch.full((len(xb),), float(sigma), device=dev); vv = v.view(1, 3, 32, 32)
            Jv += ((net(xb+eps*vv, s) - net(xb-eps*vv, s))/(2*eps)).view(len(xb), -1).sum(0)
        Jv /= n; lam = float(Jv @ v); v = Jv/Jv.norm()
    return v.cpu().numpy().astype(np.float32), lam

@torch.no_grad()
def net_binned_v(v, mid_v, half_v, tau, n=8000, nbins=50, bs=250):
    vt = torch.from_numpy(v).to(dev); sigma = float(np.clip(half_v*np.sqrt(tau), net.sigma_min, net.sigma_max))
    x0 = torch.from_numpy(X[rng.choice(len(X), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0); D = []
    for xb in xt.split(bs): D.append(net(xb.view(-1, 3, 32, 32), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1))
    D = torch.cat(D); m = ((xt @ vt - mid_v)/half_v).cpu().numpy(); sm = (half_v*((D-xt) @ vt)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
    return 0.5*(edges[:-1]+edges[1:]), np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9), cnt

print("\nsoft-mode directions of the pretrained model and bimodality of the DATA along them:")
SOFT = []
for sigma in [1.0, 2.0, 4.0, 8.0, 16.0]:
    t0 = time.time(); v, lam = soft_mode(sigma); proj = X @ v; mid_v = float(proj.mean()); half_v = float(proj.std()); m0 = (proj - mid_v)/half_v
    modes_raw = n_max(stationary_points(smooth_hist(m0, 0.01))); bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t)))
    cls_sep = sorted([(round(abs(proj[Y == c].mean()-proj[Y != c].mean())/proj.std(), 2), CLASSES[c]) for c in range(10)], reverse=True)[:3]
    SOFT.append(dict(sigma=sigma, v=v, lam=lam, m0=m0, mid=mid_v, half=half_v, bj=bj))
    print(f"  sigma={sigma:5.1f}: top eigenvalue {lam:.3f}; raw-data modes along v: {modes_raw}; exact tau_bif along v: {(bj[0] if bj else float('nan')):.3f}; most separated classes {cls_sep}  ({time.time()-t0:.0f}s)")

bim = [s_ for s_ in SOFT if s_['bj']]
for r in bim:
    bjn = birth_bisect(lambda t: zero_crossings(*net_binned_v(r['v'], r['mid'], r['half'], t)), taus=np.geomspace(0.1, 3.0, 25), iters=6)
    print(f"soft mode at sigma={r['sigma']}: exact tau_bif {r['bj'][0]:.3f} (jump {r['bj'][1]:.2f}) | network tau_bif {(bjn[0] if bjn else float('nan')):.3f}")
if not bim: print("\nNo soft mode carries a bimodal data projection: commitment in this model is not along any linear pixel-space direction we can find (LDA-chosen or model-chosen).")

# projected histograms along the soft modes, for the appendix
fig, axes = plt.subplots(1, len(SOFT), figsize=(3*len(SOFT), 2.8))
for ax, r in zip(axes, SOFT): ax.hist(r['m0'], bins=120, density=True); ax.set_title(f"data along soft mode, sigma={r['sigma']}", fontsize=8)
plt.tight_layout(); plt.show()

## What to report back
**A.** The 11-direction table with Fisher separabilities and raw-projection mode counts.
**B.** The soft-mode table (eigenvalue, data bimodality, most separated classes), any fidelity lines, and the histograms.